In [ ]:
import pandas as pd

df = pd.read_csv("C:/MY FILES/3BIZ/SLIIT/Research project/Projects done/25 11 30(Merging the 2 hiring duration analysis datasets to 1)/hiring_duration_merged.csv", parse_dates=[
    "Submission Date", "Joining Date", "Sourcing Start", 
    "Interview Start", "Interview End", "Offered", "Filled"])

In [ ]:
df.columns

In [ ]:
df[["Submission Date", "Joining Date", "Sourcing Start", "Interview Start", "Interview End", "Offered", "Filled"]]

In [ ]:
order=['Job Title', 'Job Location', 'Job Time Zone', 'Consultant Name', 'Visa',
       'Consultant Location', 'Consultant Time Zone', 'Salary ($1000)', 'Relocation',
       'Sourcing Start','Submission Date', 'Interview Start', 'Interview End',
       'Offered', 'Filled','Interview Status','Hired']
df = df[order]

In [ ]:
df[['Sourcing Start','Submission Date', 'Interview Start', 'Interview End',
       'Offered', 'Filled']]

In [ ]:

# Strip whitespace from string fields
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Fix salary format — convert "$110K" → 110000
df["Salary ($1000)"] = (
    df["Salary ($1000)"]
    .str.replace("$", "", regex=False)
    .str.replace("K", "", regex=False)
    .str.replace("k", "", regex=False)
    .astype(float) * 1000
)

# Standardize Yes/No flags
df["Relocation"] = df["Relocation"].str.title()
df["Hired"] = df["Hired"].replace({"Y": "Yes", "N": "No"})  

In [ ]:
df.head(2)

In [ ]:
# Convert Date Columns to Datetime

In [ ]:

date_columns = [
    "Sourcing Start","Submission Date", 
    "Interview Start", "Interview End", "Offered", "Filled"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")  

In [ ]:
df.head(2)

In [ ]:
# Interview duration (start → end)
df["Interview_Duration_Days"] = (
    df["Interview End"] - df["Interview Start"]
).dt.days

# Time from sourcing to offer
df["Days_Sourcing_to_Offer"] = (
    df["Offered"] - df["Sourcing Start"]
).dt.days

In [ ]:
job_summary = df.groupby("Job Title").agg({
    "Salary ($1000)": ["mean", "min", "max"],
    "Hired": lambda x: (x == "Yes").sum(),
    "Consultant Name": "count"
}).rename(columns={"Consultant Name": "Total Submitted"})

In [ ]:
job_summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set Seaborn style
sns.set(style="whitegrid")

# Salary Distribution by Job Title
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x="Job Title", y="Salary ($1000)")
plt.title("Salary Distribution by Job Title")
plt.ylabel("Salary ($)")
plt.xlabel("Job Title")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

# Hired vs Not Hired by Job Title
plt.figure(figsize=(10,6))
sns.countplot(data=df, x="Job Title", hue="Hired")
plt.title("Hired vs Not Hired by Job Title")
plt.ylabel("Count")
plt.xlabel("Job Title")
plt.xticks(rotation=45)
plt.legend(title="Hired")
plt.tight_layout()
plt.show()

In [ ]:
# Relocation Requirement by Job Title
plt.figure(figsize=(10,6))
sns.countplot(data=df, x="Job Title", hue="Relocation")
plt.title("Relocation Requirement by Job Title")
plt.ylabel("Count")
plt.xlabel("Job Title")
plt.xticks(rotation=45)
plt.legend(title="Relocation")
plt.tight_layout()
plt.show()

In [ ]:
# Average Salary per Job Title
avg_salary = df.groupby("Job Title")["Salary ($1000)"].mean().sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=avg_salary.index, y=avg_salary.values, palette="viridis")
plt.title("Average Salary per Job Title")
plt.ylabel("Average Salary ($)")
plt.xlabel("Job Title")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Time-to-Fill Analysis
# Calculate time-to-fill in days
df['Time_to_Fill'] = (df['Filled'] - df['Sourcing Start']).dt.days
plt.figure(figsize=(10,5))
sns.boxplot(x='Job Title', y='Time_to_Fill', data=df)
plt.title('Time-to-Fill by Job Title')
plt.xticks(rotation=45)
plt.show()

In [ ]:

# Salary Analysis
# Remove $ and K from Salary column and convert to numeric
df['Salary ($1000)'] = df['Salary ($1000)'].replace('[\$,K]', '', regex=True).astype(float)

# Average, min, max salary per job title
salary_stats = df.groupby('Job Title')['Salary ($1000)'].agg(['mean', 'min', 'max']).reset_index()
print("Salary Analysis per Job Title:")
print(salary_stats)


In [ ]:
df_hired = df[df['Hired'] == 'Yes']

In [ ]:
# Visa Type Analysis
visa_counts = df_hired['Visa'].value_counts()
plt.figure(figsize=(6,4))
sns.barplot(x=visa_counts.index, y=visa_counts.values)
plt.title('Number of Hires by Visa Type')
plt.ylabel('Number of Hires')
plt.show()

In [ ]:
# Relocation Analysis
relocation_counts = df_hired['Relocation'].value_counts()
plt.figure(figsize=(6,4))
sns.barplot(x=relocation_counts.index, y=relocation_counts.values)
plt.title('Number of Hires with Relocation')
plt.ylabel('Number of Hires')
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=['Job Time Zone', 'Consultant Time Zone','Relocation'])

In [ ]:
df.isnull().sum()

In [ ]:
other_date_cols = ["Interview_Duration_Days","Days_Sourcing_to_Offer","Time_to_Fill"]
df = df.drop(columns=other_date_cols)
df

In [ ]:
df.isnull().sum()

In [ ]:
# Ensure datetime dtype
date_cols = ["Sourcing Start","Submission Date", "Interview Start", "Interview End", "Offered", "Filled"]
df[date_cols] = df[date_cols].apply(pd.to_datetime)

In [ ]:
df

In [ ]:
df[["Sourcing Start","Submission Date", "Interview Start", "Interview End", "Offered", "Filled"]]

In [ ]:
df["Dur_Sourcing_to_Submission"] = df["Submission Date"] - df["Sourcing Start"]
df

In [ ]:
df["Dur_Submission_to_InterviewStart"] = df["Interview Start"] - df["Submission Date"]
df



In [ ]:
df["Dur_InterviewStart_to_InterviewEnd"] = df["Interview End"] - df["Interview Start"]
df

In [ ]:
df["Dur_InterviewEnd_to_Offered"] = df["Offered"] - df["Interview End"]
df

In [ ]:
df["Dur_Offered_to_Filled"] = df["Filled"] - df["Offered"]
df

In [ ]:
categorical = ["Job Title","Job Location","Visa","Consultant Location",
               "Job Time Zone","Consultant Time Zone","Relocation","Interview Status"]
df = pd.get_dummies(df, columns=categorical)

In [ ]:
df

In [ ]:
df[["Sourcing Start","Submission Date", "Interview Start", "Interview End", "Offered", "Filled","Dur_Sourcing_to_Submission","Dur_Submission_to_InterviewStart","Dur_InterviewStart_to_InterviewEnd","Dur_InterviewEnd_to_Offered","Dur_Offered_to_Filled"]]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X = df[feature_columns]
y = df["Dur_InterviewStart_to_InterviewEnd"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)